# 04. Embedding Model Comparison

**Question**: does switching from `multilingual-e5-base` to a
Japanese-specialized or larger model materially improve retrieval on
Japanese technical docs?

**Caveat** — cosine score distributions differ per model, so a
**fixed** threshold cannot fairly compare them. We therefore report
each model's **own best F1 at its own per-model best threshold**.

Models:
- `intfloat/multilingual-e5-base` (768d) — agent-kms default
- `intfloat/multilingual-e5-large` (1024d) — same family, larger
- `cl-nagoya/ruri-v3-310m` (768d) — Japanese-specialized, current generation
- `cl-nagoya/ruri-v3-30m` (256d) — Japanese-specialized, very small

We initially planned to test `cl-nagoya/ruri-large` and
`pkshatech/GLuCoSE-base-ja-v2`, but both depend on tokenizer / processor
classes that are not loadable in current `transformers` + `sentence-transformers`
versions (they error out at model-load time). We substitute the
`ruri-v3` line, which is the maintained successor to `ruri-large` and
covers the same "Japanese-specialized embedding" axis at two size points.

To avoid `sentence-transformers` / `qdrant-client` cache state spilling
between models, **each model is run in its own subprocess** via
`scripts/run_one_model.py`. Per-model output lands in
`results/model_<safe_name>.csv` and we aggregate here.


In [ ]:
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.eval_utils import resolve_eval_paths

CORPUS_DIR, QUERIES_PATH, COLLECTION = resolve_eval_paths(ROOT)

MODELS = [
    ("intfloat/multilingual-e5-base",   768),
    ("intfloat/multilingual-e5-large", 1024),
    ("cl-nagoya/ruri-v3-310m",          768),
    ("cl-nagoya/ruri-v3-30m",           256),
]

def safe(name): return name.replace("/", "_").replace("-", "_")


In [ ]:
results_dir = ROOT / "results"
results_dir.mkdir(exist_ok=True)
script = ROOT / "scripts" / "run_one_model.py"

for model, dim in MODELS:
    out = results_dir / f"model_{safe(model)}.csv"
    if out.exists():
        print(f"skip  {model}  (cached -> {out.name})")
        continue
    print(f"\nrun   {model}  (dim={dim})")
    t0 = time.time()
    r = subprocess.run(
        [sys.executable, str(script),
         "--model", model, "--dim", str(dim),
         "--collection", f"{COLLECTION}_{safe(model)}",
         "--out", str(out)],
        cwd=ROOT,
        capture_output=True, text=True,
    )
    elapsed = time.time() - t0
    print(r.stdout[-600:])
    if r.returncode != 0:
        print("STDERR:", r.stderr[-600:])
        raise RuntimeError(f"{model} subprocess failed")
    print(f"done in {elapsed:.1f}s -> {out.name}")


In [ ]:
frames = []
for model, dim in MODELS:
    out = results_dir / f"model_{safe(model)}.csv"
    df = pd.read_csv(out)
    frames.append(df)
all_df = pd.concat(frames, ignore_index=True)
print(f"combined: {len(all_df)} rows over {all_df['model'].nunique()} models")


In [ ]:
# Aggregate (mean over queries) per (model, threshold)
agg = (all_df.groupby(["model", "threshold"])
       .agg(precision=("precision", "mean"),
            recall=("recall", "mean"),
            f1=("f1", "mean"),
            avg_returned=("num_returned", "mean"))
       .reset_index())

# Per-model best F1 and threshold
best = agg.loc[agg.groupby("model")["f1"].idxmax()].reset_index(drop=True)

# Add ingest time (it's logged in the CSV as a constant column per model)
ingest_time = all_df.groupby("model")["ingest_time_s"].first().reset_index()
best = best.merge(ingest_time, on="model")
print("\nPer-model best F1 (with own optimal threshold):")
print(best.round(3).to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(9, 5))
palette = sns.color_palette("tab10", n_colors=len(MODELS))
for (model, _), color in zip(MODELS, palette):
    sub = agg[agg["model"] == model]
    short = model.split("/")[-1]
    ax.plot(sub["threshold"], sub["f1"], marker="o", label=short, color=color)
ax.set_xlabel("score_threshold (per-model — not directly comparable)")
ax.set_ylabel("mean F1 over in-domain queries")
ax.set_title("F1 vs threshold by embedding model")
ax.legend()
plt.tight_layout()
plt.savefig(ROOT / "results" / "fig_model_comparison.png", dpi=140, bbox_inches="tight")
plt.show()


In [ ]:
# Side-by-side bar: best F1 + ingest time + best threshold
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
best_sorted = best.sort_values("f1", ascending=False)
shorts = [m.split("/")[-1] for m in best_sorted["model"]]
ax.bar(shorts, best_sorted["f1"], color="#2ca02c")
for i, (f, t) in enumerate(zip(best_sorted["f1"], best_sorted["threshold"])):
    ax.text(i, f + 0.005, f"F1={f:.3f}\n@ T={t:.2f}", ha="center", fontsize=9)
ax.set_ylabel("best F1 (own threshold)")
ax.set_title("Per-model best F1")
ax.set_ylim(0, max(best_sorted["f1"]) * 1.25)
plt.setp(ax.get_xticklabels(), rotation=15, ha="right")

ax = axes[1]
ax.bar(shorts, best_sorted["ingest_time_s"], color="#1f77b4")
for i, t in enumerate(best_sorted["ingest_time_s"]):
    ax.text(i, t + 0.5, f"{t:.1f}s", ha="center", fontsize=9)
ax.set_ylabel("ingest time (seconds)")
ax.set_title("Per-model ingest cost (130 chunks)")
plt.setp(ax.get_xticklabels(), rotation=15, ha="right")

plt.tight_layout()
plt.savefig(ROOT / "results" / "fig_model_cost_vs_quality.png", dpi=140, bbox_inches="tight")
plt.show()

best.to_csv(ROOT / "results" / "model_comparison_summary.csv", index=False)


## Findings

Empirical per-model best F1 (with each model's own best threshold):

| model                          | best T | P     | R     | F1    | avg ret. | ingest |
|--------------------------------|--------|-------|-------|-------|----------|--------|
| **cl-nagoya/ruri-v3-310m**     | **0.84** | **0.280** | **0.667** | **0.327** | **5.4**  | **23.9s** |
| intfloat/multilingual-e5-large | 0.80   | 0.239 | 0.726 | 0.268 | 14.4     | 23.8s  |
| cl-nagoya/ruri-v3-30m          | 0.86   | 0.226 | 0.518 | 0.246 | 6.2      | 2.4s   |
| intfloat/multilingual-e5-base  | 0.82   | 0.163 | 0.423 | 0.182 | 5.3      | 7.9s   |

- **The agent-kms default model (`e5-base`) is the weakest of the four
  on this corpus** — F1 0.18 vs the best at 0.33 (≈ +80% relative
  improvement). This is a substantial gap, not measurement noise.
- **`cl-nagoya/ruri-v3-310m` wins overall** on F1, precision, and
  returned-set volume. The Japanese-specialized objective is paying for
  itself; it costs the same ingest time as `e5-large` but produces a
  cleaner, smaller returned set at the same recall point.
- **`ruri-v3-30m` is the best cost-quality trade-off**. At 256d and
  ~30M params it ingests in 2.4s — **10× faster than ruri-v3-310m or
  e5-large** — and still beats `e5-base` by 36% relative F1. For
  latency-sensitive on-device or edge deployments this is the model
  to try first.
- **Best thresholds span 0.80 – 0.86 across these four models** —
  the `0.93` default is not even close to optimal on *any* of them
  for this corpus. Anyone porting agent-kms must calibrate per
  (corpus, model) pair; copying thresholds across models is a trap.

**Recommendations for agent-kms**:
1. **Reconsider the default model.** `multilingual-e5-base` was a sensible
   choice in 2024 but the JP-specialized landscape has moved.
   `cl-nagoya/ruri-v3-310m` is a strict improvement on Japanese-heavy
   corpora at the same vector dimensionality (768d).
2. **Ship a `calibrate-threshold` CLI subcommand** that, given a small
   labelled set (e.g. 20 query → expected-source pairs), sweeps
   thresholds and proposes the F1-optimal one for the user's specific
   corpus + model.
3. **Document the 0.93 default's corpus-dependence prominently** —
   on this study's corpus it would silently return zero results for
   every query, and a user would have no way to know that without
   a calibration step.
